# 07 · Decision trees, random forests, and boosting

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Compare nonlinear models; inspect overfitting; compare against logistic regression.


## Step 1 — Prepare fixed train/test data

Keep splits and random seeds fixed for a fair comparison.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)


**Check your result:** The same split will be reused for every estimator.


## Step 2 — Fit multiple model families

A deep tree can memorize rows. Forests average many trees; boosting fits sequential corrections.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
models = {'logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)),
 'deep_tree':DecisionTreeClassifier(random_state=42),
 'forest':RandomForestClassifier(n_estimators=80,min_samples_leaf=3,random_state=42,n_jobs=1),
 'boosting':HistGradientBoostingClassifier(max_iter=80,max_leaf_nodes=15,random_state=42)}
for name,m in models.items():
    m.fit(X_train,y_train)
    print(name, 'train AUC:', round(roc_auc_score(y_train,m.predict_proba(X_train)[:,1]),3),
          'test AUC:', round(roc_auc_score(y_test,m.predict_proba(X_test)[:,1]),3))


**Check your result:** The deep tree commonly has a large train/test gap.


## Step 3 — Check deterministic inference

Repeated inference must not change predictions when the input and fitted model are unchanged.


In [ ]:
p1 = models['forest'].predict_proba(X_test[:5])
p2 = models['forest'].predict_proba(X_test[:5])
assert np.allclose(p1,p2)
print(p1)


**Check your result:** Probabilities match exactly within floating-point tolerance.


## Practice

Restrict tree max_depth to 3, 5, and 10 using training validation splits. Compare the train/validation gap.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Smaller depth reduces flexibility. Select depth on validation performance, then assess a fresh final test once.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
